# Course 16: Algorithms Behind Modern AI
## Session 12: Randomized algorithms for data pipelines, and the capstone

**Goal:** sample uniformly from a stream, shuffle correctly, measure what a shuffle buffer does to sorted data, draw from a large discrete distribution in constant time, and start the capstone plan.

**How to use this notebook:** run the cells from top to bottom. Cells marked `TODO` are for you to complete. The lab guide, quiz and assignment for this session are in the course document.

**Testing note:** every cell without a NOT RUN label was run once on a CPU (November 2026) on small built-in or synthetic data. Cells labelled NOT RUN need a library, a dataset download or a GPU; run them on Kaggle or Colab.

### 1. By hand first: keep one item from a stream of four
A data pipeline often reads a **stream**: a file too large for memory, or a log that is still growing. We want one item chosen uniformly at random, but we see the items one at a time and we do not know how many will come. **Reservoir sampling** with a reservoir of one item does it in one pass:

* keep the first item;
* when item number $i$ arrives ($i = 2, 3, 4, \dots$), replace the kept item with probability $1/i$.

Take the stream `a b c d`. On paper: what is the probability that `a` is still the kept item at the end? It must survive items 2, 3 and 4: $\frac{1}{2} \cdot \frac{2}{3} \cdot \frac{3}{4}$. Do the same for `b`, `c` and `d`. Then try the rule that looks simpler, "replace with probability 1/2 every time", and see what goes wrong.

The code below does the same calculation with exact fractions.

In [ ]:
import math, time, heapq, random, itertools
from fractions import Fraction
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

t_start = time.time()

def kept_item_probabilities(stream, replace_probability):
    # Exact distribution of the kept item after each arrival. replace_probability(i) is used for item number i (i = 2, 3, ...).
    dist = {stream[0]: Fraction(1)}
    print(f"  after {stream[0]}: " + ", ".join(f"P({x}) = {p}" for x, p in dist.items()))
    for i, item in enumerate(stream[1:], start=2):
        q = replace_probability(i)
        dist = {x: p * (1 - q) for x, p in dist.items()}     # the old item survives with probability 1 - q
        dist[item] = q                                       # the new item is taken with probability q
        print(f"  after {item}: " + ", ".join(f"P({x}) = {p}" for x, p in dist.items()))
    return dist

print("rule: replace with probability 1/i")
kept_item_probabilities("abcd", lambda i: Fraction(1, i))
print("rule: replace with probability 1/2 every time")
kept_item_probabilities("abcd", lambda i: Fraction(1, 2))

### 2. Reservoir sampling of k items
With a reservoir of $k$ items (Algorithm R; Vitter 1985): keep the first $k$ items. When item number $i > k$ arrives, draw a random integer $j$ between 1 and $i$; if $j \le k$, the new item replaces the item in place $j$. So item $i$ enters with probability $k/i$.

**Claim:** after $n$ items, every item is in the reservoir with probability $k/n$. **Proof by induction:** it is true for $n = k$. Suppose it is true after $n - 1$ items. Item $n$ enters with probability $k/n$. An older item is in the reservoir with probability $k/(n-1)$, and it stays unless item $n$ enters (probability $k/n$) and picks its place (probability $1/k$): so it stays with probability $1 - 1/n$, and $\frac{k}{n-1} \cdot \frac{n-1}{n} = \frac{k}{n}$.

**Cost:** one pass, one random number for each item, $O(n)$ time and $O(k)$ memory. The reservoir changes only about $k \ln(n/k)$ times, the same count as the replacements of the heap's root in session 7, and for the same reason. Vitter's paper shows how to jump over the items that will not enter, so that far fewer random numbers are needed.

**Test.** A stream of $n = 100$ items, $k = 10$, 20,000 runs: we count how often each position of the stream ends in the reservoir, and compare with $k/n$ by a chi-square test. One detail: the chi-square test assumes independent draws, but the 10 items of one run are 10 *different* items, so the counts vary a little less than independent draws would, by the factor $(n-k)/(n-1)$. We divide the statistic by this factor (for $k = 1$ it is 1). To see that the test can fail, we also run a version with a typical off-by-one bug: $j$ drawn between 1 and $i - 1$.

In [ ]:
def reservoir_sample(stream, k, rnd, bug=False):
    # Algorithm R. Returns the reservoir and the number of replacements.
    reservoir, replaced = [], 0
    for i, item in enumerate(stream):                        # i = 0, 1, 2, ... (item number i + 1)
        if i < k:
            reservoir.append(item)
        else:
            j = rnd.randrange(i) if bug else rnd.randrange(i + 1)   # correct: uniform on 0 .. i, so P(j < k) = k / (i + 1)
            if j < k:
                reservoir[j] = item; replaced += 1
    return reservoir, replaced

N_STREAM, K_RES, RUNS = 100, 10, 20_000
def inclusion_counts(bug, seed):
    rnd, counts, replaced = random.Random(seed), np.zeros(N_STREAM, dtype=int), 0
    for _ in range(RUNS):
        sample, r = reservoir_sample(range(N_STREAM), K_RES, rnd, bug)
        counts[sample] += 1; replaced += r
    return counts, replaced / RUNS

def chi_square_inclusion(counts):
    expected = RUNS * K_RES / N_STREAM
    plain = float(((counts - expected) ** 2 / expected).sum())
    corrected = plain * (N_STREAM - 1) / (N_STREAM - K_RES)  # the k items of one run are different items
    return plain, corrected, float(stats.chi2.sf(corrected, N_STREAM - 1))

t0 = time.perf_counter()
good, mean_replaced = inclusion_counts(bug=False, seed=0)
bad, _ = inclusion_counts(bug=True, seed=0)
seconds = time.perf_counter() - t0
expected = RUNS * K_RES / N_STREAM
print(f"stream of {N_STREAM} items, reservoir of {K_RES}, {RUNS:,d} runs; every position should be kept {expected:.0f} times (k/n = {K_RES / N_STREAM})")
for name, counts in [("Algorithm R", good), ("with the off-by-one bug", bad)]:
    plain, corrected, pval = chi_square_inclusion(counts)
    print(f"{name:24s}: counts from {counts.min()} to {counts.max()};  chi-square {plain:.1f}, corrected {corrected:.1f} with {N_STREAM - 1} degrees of freedom, p-value {pval:.3g}")
print(f"with the bug, the first 10 positions were kept {bad[:10].mean():.0f} times on average and the last 10 positions {bad[-10:].mean():.0f} times")
harmonic = lambda m: sum(1.0 / i for i in range(1, m + 1))
print(f"replacements in one run: {mean_replaced:.2f} on average;  k (H_n - H_k) = {K_RES * (harmonic(N_STREAM) - harmonic(K_RES)):.2f};  k ln(n/k) = {K_RES * math.log(N_STREAM / K_RES):.2f}")
print(f"time for {2 * RUNS:,d} runs in pure Python: {seconds:.1f} s")

plt.figure(figsize=(6.5, 3.0))
plt.plot(good / RUNS, lw=1, label="Algorithm R")
plt.plot(bad / RUNS, lw=1, label="with the off-by-one bug")
plt.axhline(K_RES / N_STREAM, color="k", ls="--", lw=0.8, label="k/n")
plt.xlabel("position in the stream"); plt.ylabel("share of runs in which it was kept")
plt.title("Reservoir sampling: every position is kept equally often", fontsize=10); plt.legend(fontsize=8)
plt.tight_layout(); plt.show()

### 3. Shuffling: Fisher–Yates against a common bug
To shuffle an array in place, the **Fisher–Yates shuffle** (in the form of Durstenfeld 1964) walks from the last position down: swap position $i$ with a position chosen uniformly from $0, \dots, i$. There are $n \cdot (n-1) \cdots 2 = n!$ equally likely sequences of choices, and each gives a different order, so every order has probability $1/n!$. Cost: $n - 1$ random numbers and swaps.

The common bug is to swap every position with a position chosen from the **whole** array. That makes $n^n$ equally likely sequences of choices. For $n = 3$ that is 27 sequences for 6 orders, and 27 is not a multiple of 6, so the orders cannot be equally likely.

Below: all 27 sequences counted exactly; then 60,000 shuffles with each method and chi-square tests; then the exact size of the bias for larger arrays.

In [ ]:
def fisher_yates(a, rnd):
    for i in range(len(a) - 1, 0, -1):
        j = rnd.randrange(i + 1)                             # uniform on 0 .. i
        a[i], a[j] = a[j], a[i]
    return a

def naive_shuffle(a, rnd):
    for i in range(len(a)):
        j = rnd.randrange(len(a))                            # the bug: any position of the array
        a[i], a[j] = a[j], a[i]
    return a

def naive_exact(n):
    # How many of the n^n equally likely sequences of choices lead to each order?
    counts = {}
    for choices in itertools.product(range(n), repeat=n):
        a = list(range(n))
        for i, j in enumerate(choices):
            a[i], a[j] = a[j], a[i]
        counts[tuple(a)] = counts.get(tuple(a), 0) + 1
    return counts

orders = list(itertools.permutations(range(3)))
exact3 = naive_exact(3)
name = lambda order: "".join("ABC"[i] for i in order)
print("naive shuffle of A B C: number of the 27 sequences of choices that give each order")
print("  " + ", ".join(f"{name(o)} {exact3[o]}" for o in orders))

N_SHUF = 60_000
rnd = random.Random(0)
freq = {"Fisher-Yates": {o: 0 for o in orders}, "naive": {o: 0 for o in orders}}
for _ in range(N_SHUF):
    freq["Fisher-Yates"][tuple(fisher_yates([0, 1, 2], rnd))] += 1
    freq["naive"][tuple(naive_shuffle([0, 1, 2], rnd))] += 1
print(f"\n{N_SHUF:,d} shuffles with each method (each order is expected {N_SHUF // 6:,d} times if the shuffle is fair)")
print("order | Fisher-Yates |  naive | naive, expected from the 27 sequences")
for o in orders:
    print(f"  {name(o)} | {freq['Fisher-Yates'][o]:12,d} | {freq['naive'][o]:6,d} | {N_SHUF * exact3[o] / 27:10.1f}")
for method in ["Fisher-Yates", "naive"]:
    chi2, pval = stats.chisquare([freq[method][o] for o in orders])
    print(f"{method:12s} against the uniform distribution: chi-square {chi2:.2f} with 5 degrees of freedom, p-value {pval:.3g}")
chi2, pval = stats.chisquare([freq["naive"][o] for o in orders], [N_SHUF * exact3[o] / 27 for o in orders])
print(f"naive        against its exact distribution:   chi-square {chi2:.2f} with 5 degrees of freedom, p-value {pval:.3g}")

print("\nthe bias does not go away for larger arrays (exact, from all n^n sequences):")
for n in [3, 4, 5, 6, 7]:
    c = naive_exact(n)
    print(f"  n = {n}: {n ** n:7,d} sequences for {math.factorial(n):5,d} orders; the most likely order is {max(c.values()) / min(c.values()):.2f} times as likely as the least likely")

### 4. A shuffle buffer on sorted data
Fisher–Yates needs the whole dataset in memory, or at least random access to it. A training set that is read as a stream (many large files, read one after another) is usually shuffled with a **shuffle buffer** instead, as in `tf.data`: fill a buffer with $B$ examples; then, again and again, output a randomly chosen example from the buffer and put the next example of the stream in its place.

What a buffer can do is limited: the example at position $i$ of the stream cannot leave before the buffer has filled, so it moves **at most $B - 1$ places earlier**. (It can come out much later.) If the stream is sorted, for example one class after another, a small buffer leaves it almost sorted.

The experiment uses scikit-learn's digits: 1,300 training images **sorted by class** (all zeros first, then all ones, ...) and 497 test images. For several buffer sizes we measure

* the largest number of places that an example moved earlier (the bound says at most $B - 1$);
* the mean number of **different classes in a batch** of 32 (a fair shuffle gives about 9.6);
* the test accuracy of a linear classifier (`SGDClassifier` with scikit-learn's default settings) trained in **one pass** over the stream, in batches of 32, with `partial_fit`. We set `shuffle=False`, so the model sees the examples in exactly the order that the buffer gives.

Each buffer size is run with 5 seeds. The model is small and one pass is short: the accuracies say something about the order of the data, not about how well digits can be classified.

In [ ]:
from sklearn.datasets import load_digits
from sklearn.linear_model import SGDClassifier
from sklearn.model_selection import train_test_split

def shuffle_buffer(stream, buffer_size, rnd):
    # Yields the items of the stream in a partly shuffled order, holding at most buffer_size items.
    buffer = []
    for item in stream:
        if len(buffer) < buffer_size:
            buffer.append(item)                              # fill the buffer first
        else:
            j = rnd.randrange(buffer_size)
            yield buffer[j]                                  # a random item leaves
            buffer[j] = item                                 # the new item takes its place
    rnd.shuffle(buffer)                                      # the stream has ended: empty the buffer in random order
    yield from buffer

digits = load_digits()
X_tr, X_te, y_tr, y_te = train_test_split(digits.data / 16.0, digits.target, test_size=497, random_state=0, stratify=digits.target)
by_class = np.argsort(y_tr, kind="stable")
X_sorted, y_sorted = X_tr[by_class], y_tr[by_class]          # the training stream: sorted by class
N_TRAIN, BATCH, SEEDS = len(y_sorted), 32, [0, 1, 2, 3, 4]
print(f"{N_TRAIN} training images sorted by class, {len(y_te)} test images; examples in each class: {np.bincount(y_sorted).tolist()}")

def one_pass_model(order):
    clf = SGDClassifier(shuffle=False, random_state=0)       # a linear classifier, scikit-learn's default settings
    for start in range(0, N_TRAIN, BATCH):
        idx = order[start:start + BATCH]
        clf.partial_fit(X_sorted[idx], y_sorted[idx], classes=np.arange(10))
    return clf

def one_pass_accuracy(order):
    return one_pass_model(order).score(X_te, y_te)

def classes_in_a_batch(order):
    return float(np.mean([len(set(y_sorted[order[s:s + BATCH]].tolist())) for s in range(0, N_TRAIN, BATCH)]))

buffer_sizes = [1, 8, 32, 128, 512, N_TRAIN]
rows, example_orders = [], {}
t0 = time.perf_counter()
print("buffer size | largest move to an earlier place (bound B - 1) | classes in a batch | test accuracy: mean (lowest to highest of 5 seeds)")
for size in buffer_sizes:
    moved, mixed, acc = [], [], []
    for seed in SEEDS:
        order = np.array(list(shuffle_buffer(range(N_TRAIN), size, random.Random(seed))))
        assert sorted(order.tolist()) == list(range(N_TRAIN))               # every example comes out exactly once
        moved.append(int((order - np.arange(N_TRAIN)).max()))               # input position minus output position
        mixed.append(classes_in_a_batch(order)); acc.append(one_pass_accuracy(order))
        if seed == 0:
            example_orders[size] = order
    rows.append((size, max(moved), float(np.mean(mixed)), float(np.mean(acc)), min(acc), max(acc)))
    label = f"{size} (all)" if size == N_TRAIN else str(size)
    print(f"{label:>11s} | {max(moved):22d} ({size - 1:4d})                  | {np.mean(mixed):18.2f} | {np.mean(acc):.3f} ({min(acc):.3f} to {max(acc):.3f})")
print(f"time for the {len(buffer_sizes) * len(SEEDS)} training passes: {time.perf_counter() - t0:.1f} s")
predicted = np.bincount(one_pass_model(example_orders[1]).predict(X_te), minlength=10)
print(f"after the sorted stream (buffer of 1), the predictions for the {len(y_te)} test images, by class 0 to 9: {predicted.tolist()}")

fig, ax = plt.subplots(1, 3, figsize=(9, 2.8), sharey=True)
for a, size in zip(ax, [1, 128, N_TRAIN]):
    a.plot(y_sorted[example_orders[size]], ".", ms=2)
    a.set_title(f"buffer of {size}" + (" (the whole set)" if size == N_TRAIN else ""), fontsize=10); a.set_xlabel("position in the output")
ax[0].set_ylabel("class of the example")
plt.tight_layout(); plt.show()

fig, ax = plt.subplots(1, 2, figsize=(9, 3.0))
sizes = [r[0] for r in rows]
ax[0].plot(sizes, [r[2] for r in rows], "o-"); ax[0].set_xscale("log", base=2)
ax[0].set_xlabel("buffer size B"); ax[0].set_ylabel("classes in a batch of 32"); ax[0].set_title("Mixing", fontsize=10)
ax[1].plot(sizes, [r[3] for r in rows], "o-")
ax[1].fill_between(sizes, [r[4] for r in rows], [r[5] for r in rows], alpha=0.2)
ax[1].set_xscale("log", base=2); ax[1].set_xlabel("buffer size B"); ax[1].set_ylabel("test accuracy after one pass")
ax[1].set_title("One pass of SGD over the buffered stream", fontsize=10)
plt.tight_layout(); plt.show()

### 5. Sampling from a discrete distribution: scan, binary search, alias
Many training loops draw an item from a fixed distribution over $V$ items millions of times. The best-known case is **negative sampling** in word2vec: "noise" words are drawn in proportion to the word count raised to the power 3/4 (Mikolov et al. 2013). Three ways to draw one item with probabilities $p_1, \dots, p_V$ from a uniform random number $u$:

* **linear scan**: add up $p_1 + p_2 + \dots$ until the sum passes $u$. No set-up, $O(V)$ for each sample in the worst case;
* **binary search** on the cumulative sums: $O(V)$ set-up once, then $O(\log V)$ for each sample (`np.searchsorted` does this);
* the **alias method** (Walker 1977; Vose 1991 gave the $O(V)$ set-up used here): cut the distribution into $V$ columns of equal width $1/V$, where each column holds at most two items, its own and one "alias". A sample picks a column uniformly, then one of its two items with a biased coin: $O(1)$ for each sample, whatever $V$ is.

How the table is built: multiply every probability by $V$. Items below 1 are "small", the others "large". Take one small and one large item: the small one keeps its own column and fills the rest of it with a piece of the large one (its alias); the large item loses that piece and may become small. Each step finishes one column, so there are at most $V$ steps.

We count the **work for one sample**: probabilities examined by the scan, probes of the binary search, and table rows read by the alias method. The distribution is Zipf-like: counts proportional to $1/r$ for the word of rank $r$, raised to the power 3/4, in the order of the ranks (most frequent first, which is the best order for the scan). The data is synthetic.

In [ ]:
def sample_scan(p, u):
    total = 0.0
    for i, pi in enumerate(p):
        total += pi
        if u < total:
            return i, i + 1                                  # the item, and how many probabilities we looked at
    return len(p) - 1, len(p)

def sample_bisect(cdf, u):
    lo, hi, probes = 0, len(cdf) - 1, 0
    while lo < hi:                                           # the first position whose cumulative sum is above u
        mid = (lo + hi) // 2; probes += 1
        if u < cdf[mid]:
            hi = mid
        else:
            lo = mid + 1
    return lo, probes

def build_alias(p):
    n = len(p)
    scaled = [x * n for x in p]
    prob, alias, steps = [1.0] * n, list(range(n)), 0
    small = [i for i, x in enumerate(scaled) if x < 1.0]
    large = [i for i, x in enumerate(scaled) if x >= 1.0]
    while small and large:
        s, l = small.pop(), large.pop(); steps += 1
        prob[s], alias[s] = scaled[s], l                     # column s: item s with probability scaled[s], else item l
        scaled[l] -= 1.0 - scaled[s]                         # item l gave away the rest of column s
        (small if scaled[l] < 1.0 else large).append(l)
    return prob, alias, steps                                # columns that are left hold only their own item (probability 1)

def sample_alias(prob, alias, u1, u2):
    i = int(u1 * len(prob))                                  # a column, chosen uniformly
    return i if u2 < prob[i] else alias[i]                   # one table row read, one comparison

def noise_distribution(V):
    w = (1.0 / np.arange(1, V + 1)) ** 0.75                  # Zipf counts to the power 3/4
    return w / w.sum()

rng = np.random.default_rng(0)
print("work for one sample (exact counts, averaged over the samples drawn)")
print("       V | scan: probabilities examined (expected value) | binary search: probes (log2 V) | alias: rows read | alias set-up steps")
work = {}
for V, n_scan in [(100, 2000), (1000, 2000), (10_000, 500), (100_000, 100)]:
    p = noise_distribution(V)
    p_list, cdf = p.tolist(), list(itertools.accumulate(p.tolist()))
    prob, alias, steps = build_alias(p_list)
    rebuilt = np.array(prob)                                 # the probability that the table gives to every item
    np.add.at(rebuilt, alias, 1.0 - np.array(prob))
    assert np.allclose(rebuilt / V, p, rtol=1e-9, atol=1e-15)                # the table encodes exactly p
    us = rng.random(2000).tolist()
    scans = [sample_scan(p_list, u) for u in us[:n_scan]]
    bisects = [sample_bisect(cdf, u) for u in us]
    assert [s[0] for s in scans] == [b[0] for b in bisects[:n_scan]]         # same u, same item
    work[V] = (np.mean([s[1] for s in scans]), float((p * np.arange(1, V + 1)).sum()), np.mean([b[1] for b in bisects]))
    print(f"{V:8,d} | {work[V][0]:14.1f} ({work[V][1]:10.1f}) from {n_scan:5,d} samples | {work[V][2]:13.2f} ({math.log2(V):5.2f})       | {1:16d} | {steps:10,d}")

# the alias method gives the right distribution: chi-square test on V = 1,000
V = 1000
p = noise_distribution(V)
prob, alias, _ = build_alias(p.tolist())
prob, alias = np.array(prob), np.array(alias)
N_DRAWS = 1_000_000
def alias_draws(n):
    i = rng.integers(V, size=n)
    return np.where(rng.random(n) < prob[i], i, alias[i])
counts = np.bincount(alias_draws(N_DRAWS), minlength=V)
chi2, pval = stats.chisquare(counts, N_DRAWS * p)
print(f"\nalias method, V = {V:,d}, {N_DRAWS:,d} draws: chi-square {chi2:.1f} with {V - 1} degrees of freedom, p-value {pval:.3f} (smallest expected count {N_DRAWS * p.min():.0f})")
zipf = 1.0 / np.arange(1, V + 1); zipf /= zipf.sum()
print(f"what the power 3/4 does (V = {V:,d}): the 10 most frequent words get {zipf[:10].sum():.3f} of the draws by plain counts and {p[:10].sum():.3f} after the power; "
      f"the rarer half of the words gets {zipf[V // 2:].sum():.3f} and {p[V // 2:].sum():.3f}")

def median_ms(fn, reps=5):
    ts = []
    for _ in range(reps):
        t0 = time.perf_counter(); fn(); ts.append(time.perf_counter() - t0)
    return 1e3 * float(np.median(ts))
for V in [1000, 100_000]:
    p = noise_distribution(V); cdf = np.cumsum(p)
    prob, alias, _ = build_alias(p.tolist()); prob, alias = np.array(prob), np.array(alias)
    print(f"time, V = {V:7,d}, 1,000,000 draws with np.searchsorted on the cumulative sums: {median_ms(lambda: np.searchsorted(cdf, rng.random(N_DRAWS), side='right')):6.1f} ms")
    print(f"time, V = {V:7,d}, 1,000,000 draws with the alias table (NumPy):               {median_ms(lambda: alias_draws(N_DRAWS)):6.1f} ms")
    print(f"time, V = {V:7,d}, 1,000,000 draws with rng.choice(V, p=p):                    {median_ms(lambda: rng.choice(V, size=N_DRAWS, p=p)):6.1f} ms")

plt.figure(figsize=(6.5, 3.2))
Vs = sorted(work)
plt.loglog(Vs, [work[v][0] for v in Vs], "s-", label="linear scan: probabilities examined")
plt.loglog(Vs, [work[v][2] for v in Vs], "o-", label="binary search: probes")
plt.loglog(Vs, [1] * len(Vs), "^-", label="alias method: rows read")
plt.xlabel("number of items V"); plt.ylabel("work for one sample")
plt.title("Sampling from a discrete distribution", fontsize=10); plt.legend(fontsize=8)
plt.tight_layout(); plt.show()

### 6. When the weights change: a sum tree
The alias table is built once. If one weight changes, the whole table must be built again: $O(V)$. Some algorithms change a weight after **every** draw. Prioritized experience replay is the best-known case: a reinforcement-learning agent draws stored transitions in proportion to a priority, and gives each drawn transition a new priority (Schaul et al. 2016).

A **sum tree** handles both operations in $O(\log n)$. It is a complete binary tree stored in an array, like a heap (session 7), but every parent holds the **sum** of its two children. The leaves hold the weights and the root holds the total.

* **update** a weight: write the leaf, then recompute the sums on the path to the root;
* **sample**: draw $u$ uniformly between 0 and the total, and walk down from the root: if $u$ is smaller than the left child's sum, go left; if not, subtract that sum from $u$ and go right.

We check the samples against the weights with a chi-square test, then run 20,000 rounds of "sample an item, then change its weight", and count the tree nodes touched.

In [ ]:
class SumTree:
    def __init__(self, n):                                   # n leaves; n must be a power of two
        self.n, self.tree, self.touched = n, [0.0] * (2 * n), 0   # tree[1] is the root, tree[n + i] is leaf i
    def update(self, i, weight):
        pos = self.n + i
        self.tree[pos] = weight; self.touched += 1
        pos //= 2
        while pos >= 1:                                      # recompute the sums on the way up
            self.tree[pos] = self.tree[2 * pos] + self.tree[2 * pos + 1]; self.touched += 1
            pos //= 2
    def total(self):
        return self.tree[1]
    def sample(self, u):                                     # u uniform on [0, total)
        pos = 1
        while pos < self.n:
            self.touched += 1
            left = self.tree[2 * pos]
            if u < left:
                pos = 2 * pos
            else:
                u -= left; pos = 2 * pos + 1
        return pos - self.n

N_LEAVES, N_TREE = 1024, 200_000
def tree_test(seed):
    # Random weights (eight of them 0), a tree over them, N_TREE samples, and a chi-square test against the weights.
    rng = np.random.default_rng(seed)
    weights = rng.uniform(0.5, 1.5, N_LEAVES)
    weights[:8] = 0.0                                        # eight items that must never be drawn
    tree = SumTree(N_LEAVES)
    for i, w in enumerate(weights):
        tree.update(i, float(w))
    update_touched, tree.touched = tree.touched // N_LEAVES, 0
    us = rng.random(N_TREE) * tree.total()
    samples = np.array([tree.sample(u) for u in us.tolist()])
    counts = np.bincount(samples, minlength=N_LEAVES)
    live = weights > 0
    chi2, pval = stats.chisquare(counts[live], N_TREE * weights[live] / weights.sum())
    same = int((samples == np.searchsorted(np.cumsum(weights), us, side="right")).sum())   # the answer of a binary search
    return {"tree": tree, "weights": weights, "chi2": chi2, "pval": pval, "df": int(live.sum()) - 1, "same": same,
            "zero_draws": int(counts[~live].sum()), "update_touched": update_touched, "sample_touched": tree.touched / N_TREE}

# 1. fixed weights: do the samples follow the weights?
r = tree_test(seed=0)
tree, weights = r["tree"], r["weights"]
print(f"{N_LEAVES:,d} leaves; root {tree.total():.6f}, sum of the weights {weights.sum():.6f}; nodes touched by one update: {r['update_touched']}")
print(f"{N_TREE:,d} samples: nodes touched for each sample {r['sample_touched']:.0f} (log2 n = {int(math.log2(N_LEAVES))});  draws of the eight items with weight 0: {r['zero_draws']}")
print(f"the tree returns the same leaf as a binary search on the cumulative sums for {r['same']:,d} of {N_TREE:,d} values of u")
print(f"chi-square against the weights: {r['chi2']:.1f} with {r['df']} degrees of freedom, p-value {r['pval']:.3f}")
repeat = [tree_test(seed)["pval"] for seed in [1, 2, 3, 4, 5]]
print("the same test with seeds 1 to 5 (new weights and new random numbers): p-values " + ", ".join(f"{v:.3f}" for v in repeat))

# 2. weights that change after every draw, as in prioritized replay
ROUNDS = 20_000
tree.touched = 0
rnd = random.Random(0)
for _ in range(ROUNDS):
    i = tree.sample(rnd.random() * tree.total())
    weights[i] = rnd.uniform(0.5, 1.5)                       # the drawn item gets a new priority
    tree.update(i, float(weights[i]))
print(f"{ROUNDS:,d} rounds of sample + update: {tree.touched / ROUNDS:.0f} nodes touched in each round; building an alias table again would cost about {N_LEAVES:,d} steps in each round")
print(f"root after the updates {tree.total():.9f}; sum of the weights {weights.sum():.9f}; equal (np.isclose): {bool(np.isclose(tree.total(), weights.sum()))}")

### 7. Weighted sampling from a stream is Gumbel-top-k
Last problem: draw $k$ **different** items from a stream, where item $i$ has a weight $w_i$, as if we drew them one after another without replacement, each time in proportion to the weights of the items that are left. Efraimidis & Spirakis (2006) showed that one pass is enough: give every item the key

$$\text{key}_i = u_i^{1/w_i}, \qquad u_i \text{ uniform on } (0, 1),$$

and keep the $k$ items with the **largest keys**, with a min-heap of size $k$ as in session 7. The weights do not have to add up to 1, and the length of the stream does not have to be known.

This is the Gumbel-top-k trick of session 7 in another form. Take the function $-\ln(-\ln x)$ of the key. It is an increasing function, so it does not change which keys are the largest, and

$$-\ln\!\big(-\ln u_i^{1/w_i}\big) = \ln w_i - \ln(-\ln u_i) = \ln w_i + \text{Gumbel noise}.$$

We check three things on 8 items with the weights 30, 22, 16, 12, 8, 6, 4, 2 (session 7 used the same numbers as probabilities) and $k = 3$: both methods choose the same items when they get the same random numbers; the ordered triples follow the exact probabilities of drawing without replacement (chi-square test); and the heap version for streams agrees with the array version.

In [ ]:
def weighted_reservoir(stream, k, uniforms):
    # stream yields (item, weight); uniforms yields one uniform random number for each item.
    heap = []                                                # min-heap of (key, item): the root is the smallest of the k largest keys
    for (item, w), u in zip(stream, uniforms):
        key = u ** (1.0 / w)
        if len(heap) < k:
            heapq.heappush(heap, (key, item))
        elif key > heap[0][0]:
            heapq.heapreplace(heap, (key, item))
    return [item for key, item in sorted(heap, reverse=True)]    # largest key first

w = np.array([30.0, 22.0, 16.0, 12.0, 8.0, 6.0, 4.0, 2.0])
p, n_items, k, N = w / w.sum(), len(w), 3, 200_000
rng = np.random.default_rng(0)
U = rng.random((N, n_items))                                 # the same random numbers for both methods

keys = U ** (1.0 / w)                                        # Efraimidis & Spirakis
top_keys = np.argsort(-keys, axis=1)[:, :k]
scores = np.log(w) - np.log(-np.log(U))                      # Gumbel-top-k: log weight plus Gumbel noise
top_gumbel = np.argsort(-scores, axis=1)[:, :k]
print(f"same three items in the same order from both methods: {int((top_keys == top_gumbel).all(axis=1).sum()):,d} of {N:,d} runs")

exact_seq, exact_incl = {}, np.zeros(n_items)                # drawing without replacement, by enumeration
for tup in itertools.permutations(range(n_items), k):
    prob, rest = 1.0, 1.0
    for i in tup:
        prob *= p[i] / rest; rest -= p[i]
    exact_seq[tup] = prob; exact_incl[list(tup)] += prob
triple_prob = np.array(list(exact_seq.values()))
triple_code = np.array([a * 64 + b * 8 + c for a, b, c in exact_seq])
observed = np.bincount(top_keys[:, 0] * 64 + top_keys[:, 1] * 8 + top_keys[:, 2], minlength=512)[triple_code]
chi2, pval = stats.chisquare(observed, N * triple_prob)
print(f"ordered triples against drawing without replacement: chi-square {chi2:.1f} with {len(exact_seq) - 1} degrees of freedom, p-value {pval:.3f} "
      f"(smallest expected count {N * triple_prob.min():.1f})")
incl = np.bincount(top_keys.ravel(), minlength=n_items) / N
print("item | weight | P(item is among the 3): exact | weighted reservoir | k * weight / total (wrong)")
for i in range(n_items):
    print(f"{i:4d} | {w[i]:6.0f} | {exact_incl[i]:29.4f} | {incl[i]:18.4f} | {k * p[i]:10.2f}")
print(f"largest difference from the exact value: {np.abs(incl - exact_incl).max():.4f}")

same = sum(weighted_reservoir(zip(range(n_items), w.tolist()), k, U[r].tolist()) == top_keys[r].tolist() for r in range(2000))
print(f"the heap version for streams gives the same answer as the array version in {same:,d} of 2,000 runs")

### 8. The same buffer in a real library (NOT RUN here)
`tf.data` is TensorFlow's input pipeline. Its `shuffle` step is the buffer of section 4; the argument `buffer_size` is our $B$. The cell is written from the documented interface and is not run here, because TensorFlow is not one of this course's libraries. PyTorch's `DataLoader(shuffle=True)` works differently for a dataset with random access: it draws a full random permutation of the indices, which is the Fisher–Yates idea. Check both libraries' documentation before you rely on the details.

In [ ]:
# NOT RUN: needs tensorflow (a library that this course does not install)
import tensorflow as tf

ds = tf.data.Dataset.from_tensor_slices((X_sorted, y_sorted))        # the stream sorted by class, as in section 4
for buffer_size in [1, 128, len(y_sorted)]:
    batches = ds.shuffle(buffer_size=buffer_size, seed=0).batch(32)
    classes = [len(set(labels.numpy().tolist())) for _, labels in batches]
    print(buffer_size, sum(classes) / len(classes))                  # mean number of classes in a batch

### 9. Exercises and the capstone plan

In [ ]:
# TODO: shuffle the shards first. Real pipelines store the data in many files (shards) and shuffle on two levels:
# first the order of the files, then a buffer over the examples. Cut the sorted stream of section 4 into 20 shards
# of 65 examples. Write shard_then_buffer(buffer_size, rnd): shuffle the order of the shards with fisher_yates,
# read them one after another, and pass the examples through shuffle_buffer. Report the classes in a batch and the
# one-pass accuracy for buffer sizes 32 and 128, next to the table of section 4. Then cut the stream into 100
# shards of 13 examples and report again. What does the size of a shard change?
def shard_then_buffer(buffer_size, rnd, n_shards=20):
    return None          # replace with your code: return the order as a NumPy array of N_TRAIN indices

order = shard_then_buffer(32, random.Random(0))
print("not done yet" if order is None else f"classes in a batch {classes_in_a_batch(order):.2f}, accuracy {one_pass_accuracy(order):.3f}")

In [ ]:
# TODO: your capstone plan (assignment 12). Fill in one row for each part of your search system. Use at least four
# structures from the course. For each part write: the structure, the session it comes from, its cost in Big-O
# notation WITH the case (worst, expected or amortized), the exact count you will measure to check that cost, and
# how you will judge quality. One row is filled in as an example of the format; replace it with your own.
capstone_plan = [
    {"part": "example: remove near-duplicate notes", "structure": "MinHash signatures with banding", "session": 9,
     "cost": "expected O(n) candidate pairs instead of n(n-1)/2", "count": "pairs compared against n(n-1)/2",
     "quality": "precision and recall on 50 pairs labelled by hand"},
]

def check_plan(plan):
    fields = ["part", "structure", "session", "cost", "count", "quality"]
    rows = [r for r in plan if not str(r.get("part", "")).startswith("example")]
    problems = []
    if len(rows) < 4:
        problems.append(f"{len(rows)} of at least 4 parts filled in")
    for r in rows:
        missing = [f for f in fields if not r.get(f)]
        if missing:
            problems.append(f"{r.get('part', '?')}: missing {', '.join(missing)}")
        if not any(case in str(r.get("cost", "")) for case in ("worst", "expected", "amortized")):
            problems.append(f"{r.get('part', '?')}: the cost does not say which case (worst, expected or amortized)")
    return problems

problems = check_plan(capstone_plan)
print("not done yet: " + "; ".join(problems) if problems else "plan table complete: " + ", ".join(r["part"] for r in capstone_plan))
print(f"runtime so far {time.time() - t_start:.0f} s")

### What to notice
(to be written from the run)